# convert_notebooks


In [ ]:
from pathlib import Path
import sys
_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'AGENTS.md').is_file())
if str(_root) not in sys.path: sys.path.insert(0, str(_root))
if str(_root / '.runtime') not in sys.path: sys.path.insert(0, str(_root / '.runtime'))
if str(_root / '.runtime') not in sys.path: sys.path.insert(0, str(_root / '.runtime'))
__file__ = str(_root / 'scripts/convert_notebooks.py')
__package__ = 'scripts'
__name__ = 'scripts.convert_notebooks'


In [ ]:
"""Idempotently add unconverted Python infrastructure to canonical notebooks."""
import ast
import hashlib
import json
from pathlib import Path
import nbformat as nbf

ROOT = Path(__file__).resolve().parents[1]


def convert():
    manifest_path = ROOT / 'notebooks/source_manifest.json'
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
    paths = list(ROOT.glob('*.py')) + list((ROOT / 'utils').rglob('*.py')) + list((ROOT / 'tests').glob('*.py')) + list((ROOT / 'scripts').glob('*.py'))
    for path in sorted(paths):
        if path.name in {'app.py', 'server.py', 'notebook_runtime.py'}:
            continue
        relative = path.relative_to(ROOT)
        source = path.read_text(encoding='utf-8-sig')
        if source.startswith('# Generated notebook import bridge.'):
            continue
        tree = ast.parse(source)
        lines = source.splitlines(keepends=True)
        entrypoint = tree.body[-1] if tree.body and isinstance(tree.body[-1], ast.If) and '__name__' in ast.get_source_segment(source, tree.body[-1].test) else None
        split = entrypoint.lineno - 1 if entrypoint else len(lines)
        prefix, entry = ''.join(lines[:split]), ''.join(lines[split:])
        module_name = '.'.join(relative.with_suffix('').parts)
        bootstrap = "from pathlib import Path\nimport sys\n_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'AGENTS.md').is_file())\nif str(_root) not in sys.path: sys.path.insert(0, str(_root))\n" + f"__file__ = str(_root / {relative.as_posix()!r})\n__package__ = {module_name.rpartition('.')[0]!r}\n__name__ = {module_name!r}\n"
        cells = [nbf.v4.new_markdown_cell(f'# `{relative.as_posix()}`\n\nMã nguồn chính ở notebook này. File `.py` chỉ làm cầu nối import.'), nbf.v4.new_code_cell(bootstrap, metadata={'tags': ['notebook-only']}), nbf.v4.new_code_cell(prefix)]
        if entry:
            cells.append(nbf.v4.new_raw_cell(entry, metadata={'script_entrypoint': True}))
        assert prefix + entry == source
        target = ROOT / relative.with_suffix('.ipynb')
        target.parent.mkdir(parents=True, exist_ok=True)
        digest = hashlib.sha256(source.encode()).hexdigest()
        notebook = nbf.v4.new_notebook(cells=cells, metadata={'kernelspec': {'display_name': 'Python (demo, CUDA)', 'language': 'python', 'name': 'demo'}, 'language_info': {'name': 'python'}, 'original_module': relative.as_posix(), 'source_sha256': digest})
        nbf.write(notebook, target)
        path.write_text(f'# Generated notebook import bridge. Edit {relative.with_suffix(".ipynb").as_posix()}.\nimport sys as _sys\nfrom pathlib import Path as _Path\n_sys.path.insert(0, str(_Path(__file__).resolve().parents[{len(relative.parts) - 1}]))\nfrom notebook_runtime import exec_notebook as _exec_notebook\n_exec_notebook(__file__, globals())\n', encoding='utf-8')
        manifest[relative.as_posix()] = {'notebook': target.relative_to(ROOT).as_posix(), 'sha256': digest, 'ast_equal': True}
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print(f'Canonical module notebooks: {len(manifest)}')


